# Clase 073 — Regularización: Ridge, Lasso, Elastic Net

Controlamos el overfitting en modelos lineales con penalización **L2 (Ridge)**, **L1 (Lasso)** y su combinación (**Elastic Net**). Vemos el rol de `alpha`, por qué Lasso hace selección de features y cómo tunear con CV.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

## 1. Dataset con features irrelevantes

`make_regression` con 50 features de las cuales solo 10 son informativas. Lasso debería anular gran parte de las 40 ruidosas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

X, y, coef_true = make_regression(
    n_samples=200, n_features=50, n_informative=10, noise=10,
    coef=True, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
n_informativas = int((coef_true != 0).sum())
print('features:', X.shape[1], '| informativas reales:', n_informativas)

## 2. OLS baseline

Siempre dentro de un `Pipeline` con `StandardScaler`: la penalización compara magnitudes de coeficientes, así que las features deben estar en la misma escala.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

def rmse(model, X, y):
    return np.sqrt(mean_squared_error(y, model.predict(X)))

ols = make_pipeline(StandardScaler(), LinearRegression()).fit(Xtr, ytr)
print('OLS RMSE train:', round(rmse(ols, Xtr, ytr), 3), '| test:', round(rmse(ols, Xte, yte), 3))

## 3. Ridge (L2)

Añade $\alpha\sum\beta^2$: **encoge** los coeficientes hacia cero (menor norma L2) sin anularlos.

In [ ]:
from sklearn.linear_model import Ridge

ridge = make_pipeline(StandardScaler(), Ridge(alpha=10.0)).fit(Xtr, ytr)
coef_ols = ols.named_steps['linearregression'].coef_
coef_ridge = ridge.named_steps['ridge'].coef_
print('Ridge RMSE test:', round(rmse(ridge, Xte, yte), 3))
print('norma L2 coef  OLS:', round(np.linalg.norm(coef_ols), 2),
      '| Ridge:', round(np.linalg.norm(coef_ridge), 2))
assert np.linalg.norm(coef_ridge) < np.linalg.norm(coef_ols)
print('OK: Ridge reduce la norma de los coeficientes sin anularlos')

## 4. Lasso (L1): sparsity

Añade $\alpha\sum|\beta|$: produce soluciones **esparsas**. Al subir `alpha` cada vez más coeficientes quedan exactamente en cero.

In [ ]:
from sklearn.linear_model import Lasso

for a in [0.1, 1.0, 10.0]:
    lasso = make_pipeline(StandardScaler(), Lasso(alpha=a, max_iter=10000)).fit(Xtr, ytr)
    nz = int((lasso.named_steps['lasso'].coef_ != 0).sum())
    print(f'Lasso alpha={a:5.1f} -> coef ≠ 0: {nz:2d} | RMSE test: {rmse(lasso, Xte, yte):.3f}')

lasso1 = make_pipeline(StandardScaler(), Lasso(alpha=1.0, max_iter=10000)).fit(Xtr, ytr)
n_nonzero = int((lasso1.named_steps['lasso'].coef_ != 0).sum())
assert n_nonzero < 50
print('OK: Lasso hace selección de features (muchos coeficientes en 0)')

## 5. Elastic Net (L1 + L2)

Combina ambas penalizaciones con `l1_ratio`. Útil con features correlacionadas, donde Lasso puro elegiría una al azar del grupo.

In [ ]:
from sklearn.linear_model import ElasticNet

enet = make_pipeline(StandardScaler(),
                     ElasticNet(alpha=0.5, l1_ratio=0.5, max_iter=10000)).fit(Xtr, ytr)
nz = int((enet.named_steps['elasticnet'].coef_ != 0).sum())
print('ElasticNet RMSE test:', round(rmse(enet, Xte, yte), 3), '| coef ≠ 0:', nz)

## 6. Tuning con CV y Lasso path

`LassoCV` busca el mejor `alpha` por cross-validation. El *path* muestra cómo cada coeficiente colapsa a cero al aumentar `alpha`.

In [ ]:
from sklearn.linear_model import LassoCV, lasso_path

Xtr_s = StandardScaler().fit_transform(Xtr)
lcv = LassoCV(cv=5, max_iter=10000, random_state=42).fit(Xtr_s, ytr)
print('mejor alpha (LassoCV):', round(lcv.alpha_, 4))

alphas, coefs, _ = lasso_path(Xtr_s, ytr, n_alphas=50)
fig, ax = plt.subplots(figsize=(7, 4))
for c in coefs[:15]:
    ax.plot(np.log10(alphas), c)
ax.axvline(np.log10(lcv.alpha_), color='k', ls=':', label='alpha CV')
ax.set_xlabel('log10(alpha)'); ax.set_ylabel('coeficiente'); ax.legend()
ax.set_title('Lasso path: los coeficientes colapsan a 0 al subir alpha')
plt.tight_layout(); plt.show()

## Ejercicios

1. **Recuperación de features.** Con `make_regression(n_informative=10)`, verificá que Lasso (alpha tuneado) identifica al menos 8 de las 10 informativas y anula al menos 30 de las 40 ruidosas.
2. **α extremos.** Ajustá Ridge con `alpha=0` y con `alpha=1e6`. Confirmá que el primero equivale a OLS y el segundo predice casi la media de $y$ (underfitting).
3. **RidgeCV vs LassoCV.** Compará RMSE de test de `RidgeCV(alphas=np.logspace(-3, 3, 50))` y `LassoCV(cv=5)` sobre el mismo dataset.
4. **Correlacionadas.** Duplicá una feature informativa (dos columnas casi idénticas) y compará cómo reparten el peso Lasso vs Elastic Net.

## Conclusiones

- **Ridge** encoge coeficientes (bueno con multicolinealidad); **Lasso** los anula (selección de features); **Elastic Net** combina ambos.
- El hiperparámetro `alpha` controla el trade-off sesgo-varianza: más alto ⇒ más bias, menos varianza.
- **Escalar es obligatorio** antes de regularizar, o las features de mayor escala reciben una penalización injusta.
- El mejor `alpha` se elige por **cross-validation** (`RidgeCV`/`LassoCV`/`ElasticNetCV`), nunca sobre el test set.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios de la seccion 🧪 **Ejercicios** del README. Cada bloque es autocontenido, se ejecuta **sin internet** y en pocos segundos. Intenta resolver cada ejercicio por tu cuenta antes de mirar la solucion.

### Ejercicio 1 — OLS baseline
Usamos el `make_regression` de la clase, escalado (offline en vez de California Housing).

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
sc = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)
rmse = lambda mdl, A, b: np.sqrt(mean_squared_error(b, mdl.predict(A)))
ols = LinearRegression().fit(Xtr_s, ytr)
print(f'OLS  RMSE train={rmse(ols, Xtr_s, ytr):.2f}  test={rmse(ols, Xte_s, yte):.2f}')
print('||coef||_2:', round(np.linalg.norm(ols.coef_), 1))

### Ejercicio 2 — Ridge
Encoge los coeficientes (menor norma) sin volverlos 0.

In [ ]:
from sklearn.linear_model import Ridge
ridge = Ridge(alpha=1.0).fit(Xtr_s, ytr)
print(f'Ridge RMSE test={rmse(ridge, Xte_s, yte):.2f}  ||coef||={np.linalg.norm(ridge.coef_):.1f}')

### Ejercicio 3 — Lasso
Al subir `alpha` mas coeficientes quedan exactamente en 0 (sparsity).

In [ ]:
from sklearn.linear_model import Lasso
for a in [0.1, 1.0, 10.0]:
    ls = Lasso(alpha=a, max_iter=10000).fit(Xtr_s, ytr)
    print(f'alpha={a:5.1f}  coef==0: {int((np.abs(ls.coef_) < 1e-8).sum()):2d}/50  RMSE test={rmse(ls, Xte_s, yte):.2f}')

### Ejercicio 4 — Elastic Net
Mezcla L1 (sparsity) + L2 (estabilidad con features correlacionadas).

In [ ]:
from sklearn.linear_model import ElasticNet
en = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000).fit(Xtr_s, ytr)
print(f'ElasticNet coef==0: {int((np.abs(en.coef_) < 1e-8).sum())}/50  RMSE test={rmse(en, Xte_s, yte):.2f}')

### Ejercicio 5 — Tuning con CV + Lasso path
`RidgeCV`/`LassoCV` eligen `alpha`; el path muestra coeficientes -> 0.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import RidgeCV, LassoCV, lasso_path
rcv = RidgeCV(alphas=np.logspace(-3, 3, 50)).fit(Xtr_s, ytr)
lcv = LassoCV(cv=5, max_iter=10000, random_state=0).fit(Xtr_s, ytr)
print('mejor alpha Ridge:', round(rcv.alpha_, 4), '| Lasso:', round(lcv.alpha_, 4))
alphas, coefs, _ = lasso_path(Xtr_s, ytr)
plt.figure(figsize=(7, 4))
for c in coefs[:10]:
    plt.plot(np.log10(alphas), c)
plt.xlabel('log10(alpha)'); plt.ylabel('coef'); plt.title('Lasso path')
plt.tight_layout(); plt.show()